# Decision Transformers & Nano-VLA

In this notebook we’ll build a **tiny, fully-trainable “VLA-style” policy** (Vision + Language →
Action) in a **standard open-source environment**, and we’ll train it **offline** using a
[**Decision Transformer**](https://arxiv.org/abs/2106.01345) (DT).

We’ll use a deliberately small setup so you can run everything on a laptop:

- **Environment:** `MiniGrid-GoToObject-*` (mission strings like _"go to the red key"_)
- **Data:** synthetic offline dataset collected from a **symbolic shortest-path “expert”** (plus
  injected noise)
- **Model:** a **NanoVLA-DT** (a few million params): CNN image encoder + tiny text encoder +
  GPT-style causal transformer that predicts actions conditioned on a **desired return-to-go**.


## Why Decision Transformers + VLAs?

**Decision Transformers (DTs)** treat control as sequence modeling: they take in returns-to-go,
states, and actions as tokens and learn to predict the next action. This makes them a natural fit
for **offline RL**, letting us reuse logged trajectories without extra value-learning machinery.
Because return-to-go is just another input token, we can steer behavior at inference time by dialing
the target return.

**Vision-Language-Action (VLA)** policies fuse pixels + text + control so robots can follow
instructions, generalize across tasks, and benefit from web/teleop data. Recent systems (RT-2,
OpenVLA, Octo) show that this multimodal conditioning unlocks broad skills from shared
architectures. In this notebook the "vision" is a tiny grid image and the "language" is a templated
mission, which keeps the VLA idea concrete.

Putting DTs and VLAs together gives a simple recipe for **instruction-following control trained like
supervised learning**: we can condition on desired returns, missions, and images to get flexible
behaviors with minimal on-robot interaction — exactly what we’ll build in this notebook.


## Learning topics & simplifications

This notebook treats the following topics:

1. The DT framing: **RL as conditional sequence modeling** (Return-to-Go, State, Action tokens).
2. How to build an **offline RL dataset** from trajectories.
3. How to turn a toy language-conditioned environment into a **mini "VLA"**.
4. Why **sim2real**, **sym2real**, and "learning from experience" matter in robotics.

> We covered the fundamentals of offline RL (Behavioral Cloning, IQL) in **notebook 10**. Here we
> take a completely different approach: instead of learning value functions or cloning actions, we
> frame offline RL as **sequence modeling** via Decision Transformers.

In order to keep it simple and educational, the following simplifications are in place:

- We use a **discrete** environment and **discrete actions**.
- Our "expert" has **privileged access** to the grid (symbolic planner). This is a _sym2sim_ trick
  to cheaply generate demonstrations, not something you get in the real world.
- The "VLA" here is **not** a foundation model; it's a **nano** multimodal policy meant for learning
  the core ideas.


In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

import gymnasium as gym
from tqdm.auto import trange

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

from minigrid.wrappers import RGBImgObsWrapper

from util.rl_algos import DEVICE
from util.gymnastics import init_random, gym_simulation, plan_minigrid_goto_object

In [2]:
init_random(seed=0)

## Environment: language-conditioned MiniGrid

We want something that already looks like a tiny “VLA” task:

- **Vision:** an RGB grid observation
- **Language:** a mission string (“go to the red key”)
- **Action:** discrete action space

`MiniGrid-GoToObject-*` does exactly that: at every reset, the environment samples a **target
object** and provides a mission string telling you which object to reach.


In [ ]:
ENV_ID = "MiniGrid-GoToObject-8x8-N2-v0"


def make_env(render_mode: Optional[str] = None):
    env = gym.make(ENV_ID, render_mode=render_mode)
    env = RGBImgObsWrapper(env)  # dict obs with 'image' (RGB) + 'mission' (string) + 'direction'
    return env


env = make_env()
obs, info = env.reset(seed=0)
obs.keys(), obs["mission"], env.action_space

In [ ]:
# Visualize a single observation
img = obs["image"]
plt.figure(figsize=(3, 3))
plt.imshow(img)
plt.axis("off")
plt.title(obs["mission"])
plt.show()

## A symbolic “expert” planner (sym2sim → offline dataset)

To train a Decision Transformer we need an **offline dataset of trajectories**.

In real robotics, demonstrations come from humans, teleoperation, or other policies. Here we’ll do
something simpler (and _cheaper_): use the **full grid state** to compute a shortest path to the
target object.

The `util.gymnastics.plan_minigrid_goto_object` function plans a shortest path for the GoToObject
environment and returns the action list (with a safety check on the env id).


In [ ]:
env = make_env()
obs, info = env.reset(seed=0)
plan = plan_minigrid_goto_object(env)
print(plan)

The planner returns a list of MiniGrid **discrete action indices**: 0 = turn left, 1 = turn right,
2 = move forward, 3 = pick up, 4 = drop, 5 = toggle, 6 = done. So a plan like `[0, 2, 2, 2, 0, 2,
6]` reads as: _turn left, walk three steps forward, turn left, step forward, done_.

This is a _symbolic_ (graph search) solution — useful because it:

- gives us clean trajectories quickly,
- creates a natural bridge to **sym2real** discussions later,
- lets us inject noise to produce both good and bad trajectories (so the DT can learn to condition
  on return).

> In the real world, “privileged” state is often unavailable. Replacing it with vision-only
> perception is part of the sim2real gap.


### Collecting trajectories with noise

The `rollout_with_planner` function executes the expert plan in the environment but **injects
noise**: at each step, with probability `noise_prob`, the expert action is replaced by a random one.
This is how we get a _mixture_ of trajectory qualities — some near-optimal, some messy — which is
critical for learning **value conditioning** later.

After the episode, we compute the **return-to-go** (RTG) for each timestep as the _reverse
cumulative sum_ of rewards:

$$
\hat{R}_t = \sum_{k=t}^{T} r_k
$$

This tells the model "how much total reward is still ahead from this point." During training, RTG
becomes the conditioning signal: "if you want this much future reward, take these kinds of actions."

In [ ]:
def rollout_with_planner(
    env, noise_prob: float = 0.1, max_steps: Optional[int] = None, seed: Optional[int] = None
):
    obs, info = env.reset(seed=seed)
    traj = {"images": [], "missions": [], "actions": [], "rewards": [], "dones": []}

    if max_steps is None:
        max_steps = env.unwrapped.max_steps

    expert_plan = plan_minigrid_goto_object(env)
    t = 0
    while True:
        # TODO: append obs["image"] (copy!) and obs["mission"] to the trajectory dict
        pass

        # TODO: pick action — if t < len(expert_plan) and random.random() > noise_prob,
        #       use expert_plan[t]; otherwise sample a random action from env.action_space
        action = None

        # TODO: step the environment with the chosen action
        obs, reward, terminated, truncated, info = None, 0, False, False, {}

        # TODO: append action, float(reward), and done (terminated or truncated) to traj
        pass

        t += 1
        if terminated or truncated or (t >= max_steps):
            break

    # TODO: compute returns-to-go (RTG) from the list of rewards (reverse cumsum)
    # Hint: rewards[::-1].cumsum()[::-1] gives the RTG for each timestep
    traj["rtg"] = []

    traj["len"] = len(traj["actions"])
    traj["return"] = float(sum(traj["rewards"]))
    return traj


env = make_env()
traj = rollout_with_planner(env, noise_prob=0.2, seed=0)
traj["len"], traj["return"], traj["missions"][0]

## Build an offline dataset

We’ll collect a few thousand episodes. For learning **value conditioning**, we want a _mixture_ of:

- near-expert trajectories (low noise),
- messier trajectories (higher noise).

Each trajectory stores a per-step return-to-go (reverse cumulative reward), which becomes the
conditioning signal for the DT.

**Value Conditioning** means the model doesn't just learn "the best" policy; it learns a
distribution of behaviors indexed by their performance. By seeing both clumsy (low return) and
expert (high return) trajectories, the Decision Transformer learns the causal link between "expected
return" and "actions required to achieve it". This allows us to control the agent at inference time:
**“if you ask for higher return, choose more expert-like actions.”**


In [ ]:
@dataclass
class OfflineDatasetConfig:
    n_episodes: int = 2500
    noise_prob_low: float = 0.05
    noise_prob_high: float = 0.35
    mix_prob_high_noise: float = 0.5
    max_steps: Optional[int] = None


cfg = OfflineDatasetConfig()

env = make_env()
trajectories = []

for i in trange(cfg.n_episodes):
    high_noise = random.random() < cfg.mix_prob_high_noise
    noise_prob = cfg.noise_prob_high if high_noise else cfg.noise_prob_low
    trajectories.append(rollout_with_planner(env, noise_prob=noise_prob, seed=i))

returns = np.array([t["return"] for t in trajectories], dtype=np.float32)
print("Episodes:", len(trajectories))
print(
    "Return: mean", returns.mean(), "std", returns.std(), "min", returns.min(), "max", returns.max()
)

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(returns, bins=40)
plt.title("Offline dataset return distribution")
plt.xlabel("episode return")
plt.ylabel("count")
plt.show()

## Tokenize missions (tiny text encoder)

Real VLAs use large pretrained tokenizers + language encoders. Here we’ll do something minimal and
transparent:

- split by whitespace,
- build a tiny vocab from the dataset,
- embed tokens and average-pool them.

We encode each mission once per episode and reuse it across all timesteps, since the instruction
stays constant.

Because `GoToObject` missions are templated, this is enough.


In [ ]:
def build_vocab(missions: List[str], min_freq: int = 1):
    freq = {}
    # TODO: count token frequencies — split each mission by whitespace (lowercased)
    #       and increment freq[token] for each token

    vocab = {"<pad>": 0, "<unk>": 1}
    # TODO: for each token with freq >= min_freq (sorted), assign the next integer ID

    return vocab


all_missions = [t["missions"][0] for t in trajectories]
vocab = build_vocab(all_missions)


def tokenize(mission: str) -> List[int]:
    # TODO: split mission.lower() by whitespace, map each token to its vocab ID
    #       (use vocab["<unk>"] for unknown tokens)
    return []


max_mission_len = max(len(tokenize(m)) for m in all_missions)
len(vocab), max_mission_len, all_missions[0], tokenize(all_missions[0])

## Decision Transformer for a "Nano VLA"

### DT recap

Decision Transformer turns an RL problem into supervised sequence modeling by training a causal
transformer over tokens:

$$
(\text{RTG}_1, s_1, a_1,\; \text{RTG}_2, s_2, a_2,\;\dots)
$$

**What are RTG tokens?** Return-to-Go ($\text{RTG}_t$, or $\hat{R}_t$) is the sum of _future_
rewards from step $t$ until the end of the episode: $\hat{R}_t = \sum_{k=t}^T r_k$.

- **During training:** We compute the actual $\hat{R}_t$ from the offline trajectory. The model
  learns $P(a_t | \hat{R}_t, s_t, \dots)$.
- **At inference:** We _prompt_ the model with a **desired return** (e.g., "1.0" for a perfect run).
  The model generates actions that are most likely to yield that return.

### How DT differs from other offline RL approaches

In **notebook 10** we explored two other offline RL strategies:

- **Behavioral Cloning (BC):** directly imitates the expert's actions via supervised learning, but
  it clones a _single_ behavior — there's no way to ask for better or worse performance at test
  time.
- **IQL (Implicit Q-Learning):** learns value functions from offline data and derives a policy from
  them. This is powerful but requires careful handling of out-of-distribution actions and
  bootstrapping.

Decision Transformer sidesteps value learning entirely. It frames offline RL as **conditional
sequence modeling**: the model learns to predict actions given the desired return, and the return
token acts as a "performance dial" at inference time. The training objective is just cross-entropy
on action prediction:

$$
\mathcal{L} = -\sum_{t} \mathbb{1}[\text{valid}_t] \; \log \pi_\theta(a_t \mid \hat{R}_t, s_{\le t}, a_{<t})
$$

No value loss, no policy gradient, no importance sampling — just supervised learning with an
attention mask for padded positions.

### Failure modes to keep in mind

- **Unrealistically high desired return:** if you prompt the model with a return it never saw in
  training, behavior is undefined. The model may attempt expert-like actions that don't compose well,
  or it may degrade unpredictably. DTs only interpolate within the training distribution — they don't
  extrapolate.
- **Too-narrow dataset:** if the offline data contains _only_ expert trajectories, the model never
  learns the causal link between return level and action quality. Return conditioning requires seeing
  a _range_ of performances — which is exactly why we inject noise above.

### Our NanoVLA-DT token design

Each timestep $t$ produces:

- **RTG token:** A projection of the scalar $\hat{R}_t$
- **State token:** fused embedding of (RGB image, mission text)
- **Action token:** discrete action embedding

We'll predict each action $a_t$ from the hidden state of the corresponding **state token** $s_t$
(the position right before the action token), to avoid leaking the true action during training.

### Subsequence sampling for transformer training

Transformers operate on fixed-length sequences, but our trajectories vary in length. The
`TrajectorySubsequenceDataset` handles this by:

1. **Sliding-window sampling:** for trajectories longer than `K` (the context length), we pick a
   random start index and extract a slice of exactly `K` steps. Each epoch, a different random
   window is sampled, so the model sees varied portions of the trajectory.
2. **Padding short trajectories:** if an episode is shorter than `K`, we pad images, actions, and
   RTGs with zeros and set the **attention mask** to `0` for padded positions. This tells the
   transformer to ignore those tokens during self-attention.
3. **Pre-tokenizing missions:** since the mission string stays constant throughout an episode, we
   tokenize it once in `__init__` and reuse it for every sample, avoiding redundant work in the
   training loop.

In [ ]:
class TrajectorySubsequenceDataset(Dataset):
    """Samples fixed-length trajectory slices for DT training."""

    def __init__(self, trajectories, context_len: int):
        self.trajs = trajectories
        self.K = context_len
        # TODO: pre-tokenize missions — store tokenize(t["missions"][0]) in each traj dict
        #       and compute self.max_mission_len from the longest tokenized mission

        self.max_mission_len = 0  # Placeholder

    def __len__(self):
        return len(self.trajs)

    def __getitem__(self, idx):
        traj = self.trajs[idx]
        T = traj["len"]

        # TODO: determine start index and padding needed
        #   - if T >= K: pick a random start in [0, T-K], pad = 0
        #   - if T < K:  start at 0, pad = K - T
        # Use sl = slice(start, start + actual_len) for indexing

        # TODO: slice images, actions, rtg from traj using sl
        #       Convert to numpy arrays (images: uint8, actions: int64, rtg: float32)
        images = None
        actions = None
        rtg = None

        # TODO: if pad > 0, concatenate zero arrays to reach length K
        #       e.g. np.concatenate([images, np.zeros((pad, H, W, 3), dtype=np.uint8)])

        # TODO: build mission tokens and mask
        #   - get the pre-tokenized mission as np.array (int64)
        #   - create mask = np.ones_like(mtoks, float32)
        #   - if shorter than max_mission_len, pad both with zeros
        mtoks = None
        mmask = None

        # TODO: create attention mask — np.ones((K,), float32), then set last `pad` entries to 0
        attn = None

        return {
            "images": torch.tensor(images).float() / 255.0,
            "actions": torch.tensor(actions),
            "rtg": torch.tensor(rtg),
            "attn_mask": torch.tensor(attn),
            "mission_tokens": torch.tensor(mtoks),
            "mission_mask": torch.tensor(mmask),
        }

In [ ]:
# Unit test for TrajectorySubsequenceDataset
random.seed(0)
np.random.seed(0)

K = 4
H, W = 2, 2


def _make_traj(T, mission):
    return {
        "missions": [mission],
        "len": T,
        "images": [np.full((H, W, 3), fill_value=i, dtype=np.uint8) for i in range(T)],
        "actions": list(range(T)),
        "rtg": [float(T - i) for i in range(T)],
    }


test_trajs = [
    _make_traj(6, "go left"),
    _make_traj(2, "pick up red key"),
]

ds_test = TrajectorySubsequenceDataset(test_trajs, context_len=K)

s0 = ds_test[0]
assert s0["images"].shape == (K, H, W, 3)
assert s0["actions"].shape == (K,)
assert s0["rtg"].shape == (K,)
assert s0["attn_mask"].shape == (K,)
assert s0["attn_mask"].sum().item() == K

s1 = ds_test[1]
assert s1["attn_mask"].shape == (K,)
assert s1["attn_mask"][:2].sum().item() == 2
assert s1["attn_mask"][2:].sum().item() == 0

max_mlen = max(len(tokenize(t["missions"][0])) for t in test_trajs)
assert s0["mission_tokens"].shape == (max_mlen,)
assert s1["mission_tokens"].shape == (max_mlen,)
assert int(s1["mission_mask"].sum().item()) == len(tokenize(test_trajs[1]["missions"][0]))

print("TrajectorySubsequenceDataset unit test passed.")

### Architecture details

The `NanoVLADecisionTransformer` below wires together three encoders and a causal transformer. Here
are the key design choices:

**Token interleaving.** Each timestep produces 3 tokens arranged as `[RTG, State, Action]`. Over `K`
timesteps the full sequence is `[R₁, S₁, A₁, R₂, S₂, A₂, ..., Rₖ, Sₖ, Aₖ]` — a total of `3K`
tokens. A standard **causal (upper-triangular) mask** ensures each token can only attend to itself
and earlier tokens, preserving the autoregressive property.

**State = vision + language fusion.** The `TinyImageEncoder` (a 3-layer CNN + adaptive pooling)
encodes each RGB frame into a vector. The `TinyTextEncoder` embeds and average-pools the mission
tokens. These two features are concatenated and projected into `d_model` to form the state token.
Because the mission is constant within an episode, the text feature is computed once and broadcast
across all `K` timesteps.

**Predicting from the state position.** To predict action $a_t$, we read the transformer's hidden
state at the **state token** position (index `t * 3 + 1`). This is the position _right before_ the
action token, so the model cannot cheat by seeing the true $a_t$ — it must predict it from context.

**Type + time embeddings.** Two learned embedding tables are added to the token representations:
_type embeddings_ (3 entries: RTG / state / action) tell the transformer what kind of token it's
processing, and _time embeddings_ (`K` entries) encode the timestep index. Together they supply the
positional structure that the interleaved sequence needs.

In [ ]:
class TinyTextEncoder(nn.Module):
    """Embeds and pools mission tokens into a single text feature."""

    def __init__(self, vocab_size: int, d_text: int):
        super().__init__()
        # TODO: define an nn.Embedding(vocab_size, d_text)
        self.emb = None

    def forward(self, tokens: torch.Tensor, mask: torch.Tensor):
        # tokens: (B, L)  mask: (B, L)
        # TODO: embed tokens → (B, L, d_text), then masked-average over L
        #   Hint: multiply by mask.unsqueeze(-1), sum over dim=1,
        #         divide by mask.unsqueeze(-1).sum(dim=1).clamp_min(1.0)
        return None


class TinyImageEncoder(nn.Module):
    """Lightweight CNN that encodes image frames into feature vectors."""

    def __init__(self, d_img: int):
        super().__init__()
        # TODO: define a small CNN: 3 Conv2d layers (3→32→64→64, kernel=3, stride=2, padding=1)
        #       each followed by ReLU, then AdaptiveAvgPool2d((1, 1))
        self.net = nn.Sequential()
        # TODO: define a linear projection from 64 → d_img
        self.proj = None

    def forward(self, images: torch.Tensor):
        B, K, H, W, C = images.shape
        # TODO: permute to (B*K, C, H, W), run through self.net,
        #       flatten to (B*K, 64), project with self.proj, reshape to (B, K, d_img)
        return None


class NanoVLADecisionTransformer(nn.Module):
    """Tiny DT that fuses vision + language and predicts actions."""

    def __init__(
        self,
        action_dim: int,
        vocab_size: int,
        context_len: int,
        d_model: int = 192,
        n_layers: int = 4,
        n_heads: int = 4,
        d_text: int = 64,
        d_img: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.action_dim = action_dim
        self.K = context_len
        self.d_model = d_model

        # TODO: create TinyTextEncoder and TinyImageEncoder
        self.text_enc = None
        self.img_enc = None

        # TODO: create projection layers:
        #   - state_proj: Linear(d_text + d_img, d_model)  — fuses image + text into a state token
        #   - rtg_proj:   Linear(1, d_model)               — projects scalar RTG to d_model
        #   - act_emb:    Embedding(action_dim, d_model)    — embeds discrete actions

        # TODO: create learned embeddings for the 3 token types and for timesteps:
        #   - type_emb: Embedding(3, d_model)          — 0=rtg, 1=state, 2=action
        #   - time_emb: Embedding(context_len, d_model)

        # TODO: create a TransformerEncoder (causal):
        #   - layer = nn.TransformerEncoderLayer(d_model, n_heads, 4*d_model, dropout,
        #             batch_first=True, activation="gelu")
        #   - self.tr = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.tr = None

        # TODO: create LayerNorm(d_model) and the action prediction head Linear(d_model, action_dim)
        self.ln = None
        self.act_head = None

    def forward(self, images, mission_tokens, mission_mask, rtg, actions, attn_mask):
        B, K = rtg.shape

        # TODO: encode text → (B, d_text), unsqueeze+expand to (B, K, d_text)
        #       encode images → (B, K, d_img)

        # TODO: fuse state = state_proj(cat([img, text], dim=-1))  → (B, K, d_model)
        #       rtg_tok = rtg_proj(rtg.unsqueeze(-1))              → (B, K, d_model)
        #       act_tok = act_emb(actions)                         → (B, K, d_model)

        # TODO: interleave tokens — stack [rtg_tok, state, act_tok] along dim=2
        #       then reshape to (B, 3*K, d_model)
        #       Resulting sequence: [rtg_0, s_0, a_0, rtg_1, s_1, a_1, ...]
        tokens = None

        # TODO: add type embeddings — type_ids = [0,1,2] repeated K times
        #       add time embeddings — time_ids = [0,0,0, 1,1,1, ...] (each repeated 3×)

        # TODO: create causal mask — upper-triangular bool matrix (3K × 3K), diagonal=1
        #       create padding mask — expand attn_mask with repeat_interleave(3, dim=1)

        # TODO: run transformer: h = self.tr(tokens, mask=causal, src_key_padding_mask=pad)
        #       apply LayerNorm: h = self.ln(h)
        h = None

        # TODO: extract hidden states at "state" positions (indices 1, 4, 7, ... = i*3+1)
        #       these predict the action at each timestep
        #       return self.act_head(h[:, state_positions, :])
        return None

    @torch.no_grad()
    def act(self, images, mission_tokens, mission_mask, rtg_prefix, actions_prefix, attn_prefix):
        # TODO: run self.forward(...), then return logits at the last valid timestep
        #       last_idx = int(attn_prefix.sum()) - 1
        return None

## Train the NanoVLA-DT (offline)

We’ll train purely by supervised learning on the offline dataset:

- sample subsequences of length **K**
- pad shorter episodes and use an attention mask
- predict actions
- minimize cross-entropy

This is “offline RL” in the DT sense: **no environment interaction during training**.


In [ ]:
# ADD_TODOs IN THIS CELL

CONTEXT_LEN = 32
BATCH_SIZE = 64
LR = 3e-4
EPOCHS = 6
GRAD_CLIP = 1.0

ds = TrajectorySubsequenceDataset(trajectories, context_len=CONTEXT_LEN)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)

tmp_env = make_env()
ACTION_DIM = tmp_env.action_space.n
tmp_env.close()

model = NanoVLADecisionTransformer(
    action_dim=ACTION_DIM,
    vocab_size=len(vocab),
    context_len=CONTEXT_LEN,
    d_model=192,
    n_layers=4,
    n_heads=4,
).to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)


def batch_to_device(b):
    return {k: v.to(DEVICE) for k, v in b.items()}


def train_one_epoch():
    model.train()
    losses = []
    for batch in loader:
        batch = batch_to_device(batch)
        # TODO: forward pass — call model with all batch keys
        logits = None

        # TODO: compute cross-entropy loss, masked by attn_mask
        #   Hint: use F.cross_entropy(..., reduction="none").view(-1, CONTEXT_LEN)
        #   then multiply by attn_mask and divide by mask.sum() to ignore padded steps
        loss = None

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        losses.append(loss.item())
    return float(np.mean(losses))


train_losses = []
for ep in range(EPOCHS):
    l = train_one_epoch()
    train_losses.append(l)
    print(f"epoch {ep+1}/{EPOCHS} | loss {l:.4f}")

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(train_losses)
plt.title("Training loss (offline)")
plt.xlabel("epoch")
plt.ylabel("cross-entropy")
plt.show()

## Evaluate: return conditioning

We’ll run the environment online **only for evaluation**. We compare a **high** vs **low** desired
return-to-go. At rollout time we initialize a desired return and decrement it as rewards arrive to
keep the conditioning consistent.


In [ ]:
# ADD_TODOs IN THIS CELL


@torch.no_grad()
def run_episode_dt(
    env, model, desired_return: float, seed: Optional[int] = None, max_steps: Optional[int] = None
):
    obs, info = env.reset(seed=seed)
    if max_steps is None:
        max_steps = env.unwrapped.max_steps

    # TODO: tokenize the mission string and create a mask tensor
    #   mtoks: tensor of shape (1, mission_len) on DEVICE
    #   mmask: ones_like(mtoks, float32)
    mtoks = None
    mmask = None

    K = model.K
    H, W = obs["image"].shape[:2]

    # TODO: init zero buffers on DEVICE, all with batch dim 1:
    #   images_buf: (1, K, H, W, 3)     actions_buf: (1, K) long
    #   rtg_buf:    (1, K) float         attn_buf:    (1, K) float

    total_return = 0.0
    t = 0
    rtg_remaining = desired_return

    while True:
        # TODO: rolling window — if t < K, idx = t;
        #       else shift all buffers left by 1 (buf[:, :-1] = buf[:, 1:].clone()) and idx = K-1

        # TODO: fill buffers at idx:
        #   images_buf[:, idx] = tensor(obs["image"]).float() / 255.0
        #   rtg_buf[:, idx] = rtg_remaining
        #   attn_buf[:, idx] = 1.0

        # TODO: get action — call model.act(...), then argmax
        action = None

        obs, reward, terminated, truncated, info = env.step(action)
        total_return += float(reward)

        # TODO: update rtg_remaining (subtract reward) and store action in actions_buf[:, idx]

        t += 1
        if terminated or truncated or (t >= max_steps):
            break

    return {"return": total_return, "success": bool(terminated), "steps": t}


@torch.no_grad()
def evaluate_dt(n_eval: int, desired_return: float, seed0: int = 10_000):
    env = make_env()
    stats = []
    for i in range(n_eval):
        stats.append(run_episode_dt(env, model, desired_return=desired_return, seed=seed0 + i))
    env.close()
    return stats


hi = float(np.percentile(returns, 90))
lo = float(np.percentile(returns, 10))
print("Desired return examples:", "lo", lo, "hi", hi)

In [ ]:
N_EVAL = 100
stats_hi = evaluate_dt(N_EVAL, desired_return=hi)
stats_lo = evaluate_dt(N_EVAL, desired_return=lo)


def summarize(stats):
    rets = np.array([s["return"] for s in stats], dtype=np.float32)
    succ = np.mean([s["success"] for s in stats])
    steps = np.array([s["steps"] for s in stats], dtype=np.float32)
    return {
        "return_mean": float(rets.mean()),
        "success_rate": float(succ),
        "steps_mean": float(steps.mean()),
    }


summarize(stats_lo), summarize(stats_hi)

**What do these results tell us?** The high-return policy reaches significantly higher success rates
and shorter episodes than the low-return policy — return conditioning works!

But notice something interesting: even with a desired return of **zero** (i.e., "expect no reward at
all"), the agent still succeeds a non-trivial fraction of the time. The DT can't _intentionally
fail_ — it simply takes noisier, less directed paths. This is a known limitation: return
conditioning steers the _quality_ of behavior but cannot produce adversarial or deliberately bad
actions that weren't in the training data. The model has only seen trajectories that _try_ (with
varying competence), so it has no notion of "try to lose."

### Visualize conditioning: low vs high desired return (two rollouts)

We run **two** simulations on the **same** seeded environment:

- one with a _low_ desired return
- one with a _high_ desired return

To reuse `gym_simulation`, we wrap the DTVLA/Decision Transformer policy with an `act(obs)` method,
and we wrap the env so we can feed rewards back to the agent (DT needs rewards to update the
remaining return-to-go).


### Inference-time context management

At inference time, the Decision Transformer needs the same (RTG, state, action) context window it
was trained on. We maintain a **rolling buffer** of the last `K` timesteps:

- As each new observation arrives, it's written into the buffer at the current index.
- Once `t >= K`, we shift the buffer left by one position (dropping the oldest step) and always
  write to the last slot.
- The **RTG is maintained online**: we initialize it with the desired return, and after each
  environment step we subtract the received reward — so the model always sees the "remaining" return
  it should aim for.

Because the course's `gym_simulation` utility doesn't natively pass rewards back to the agent, we
use a small `HookStepEnv` wrapper that calls `agent.after_step(reward, done)` after every
`env.step()`. The `DTVLAAgentAdapter` then uses this hook to update its internal RTG tracking.

In [16]:
class HookStepEnv(gym.Wrapper):
    """Small env wrapper that forwards (reward, done) to the agent after each env.step()."""

    def __init__(self, env: gym.Env, agent):
        super().__init__(env)
        self._agent = agent

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = bool(terminated or truncated)
        if hasattr(self._agent, "after_step"):
            self._agent.after_step(reward, done)
        return obs, reward, terminated, truncated, info


class DTVLAAgentAdapter:
    """Adapter that makes the DTVLA Decision Transformer look like a normal RL agent with `act(obs)`."""

    def __init__(self, model, desired_return: float, device=DEVICE):
        self.model = model
        self.desired_return = float(desired_return)
        self.device = device

        self.model.eval()
        self.K = int(model.K)
        self._initialized = False

    def _init_from_obs(self, obs):
        mission = obs["mission"]
        self.mtoks = torch.tensor([tokenize(mission)], device=self.device)
        self.mmask = torch.ones_like(self.mtoks, dtype=torch.float32, device=self.device)

        H, W = obs["image"].shape[:2]
        self.images_buf = torch.zeros((1, self.K, H, W, 3), device=self.device)
        self.actions_buf = torch.zeros((1, self.K), dtype=torch.long, device=self.device)
        self.rtg_buf = torch.zeros((1, self.K), dtype=torch.float32, device=self.device)
        self.attn_buf = torch.zeros((1, self.K), dtype=torch.float32, device=self.device)

        self.t = 0
        self.rtg_remaining = float(self.desired_return)
        self._initialized = True

    def act(self, obs):
        if not self._initialized:
            self._init_from_obs(obs)

        # Maintain a rolling window of length K.
        if self.t < self.K:
            idx = self.t
        else:
            self.images_buf[:, :-1] = self.images_buf[:, 1:].clone()
            self.actions_buf[:, :-1] = self.actions_buf[:, 1:].clone()
            self.rtg_buf[:, :-1] = self.rtg_buf[:, 1:].clone()
            self.attn_buf[:, :-1] = self.attn_buf[:, 1:].clone()
            idx = self.K - 1

        self.images_buf[:, idx] = torch.tensor(obs["image"], device=self.device).float() / 255.0
        self.rtg_buf[:, idx] = self.rtg_remaining
        self.attn_buf[:, idx] = 1.0

        logits = self.model.act(
            self.images_buf, self.mtoks, self.mmask, self.rtg_buf, self.actions_buf, self.attn_buf
        )
        action = int(torch.argmax(logits, dim=-1).item())

        # Store chosen action in the context immediately (reward incorporated via after_step).
        self.actions_buf[:, idx] = action
        self.t += 1
        return action

    def after_step(self, reward: float, done: bool):
        self.rtg_remaining -= float(reward)
        if done:
            # If the env runs multiple episodes with the same agent instance, reset internal buffers.
            self._initialized = False

In [ ]:
# --- Two sequential rollouts on the same seed (one under the other) ---
VIZ_SEED = 10_000  # deterministic seed for the environment

print(f"LOW desired return rollout (seed={VIZ_SEED}, desired_return={lo:.3f})")
agent_lo = DTVLAAgentAdapter(model, desired_return=lo)
gym_simulation(
    ENV_ID,
    agent=agent_lo,
    max_t=320,
    seed=VIZ_SEED,
    wrappers=[RGBImgObsWrapper, lambda env: HookStepEnv(env, agent_lo)],
)

In [ ]:
print(f"HIGH desired return rollout (seed={VIZ_SEED}, desired_return={hi:.3f})")
agent_hi = DTVLAAgentAdapter(model, desired_return=hi)
gym_simulation(
    ENV_ID,
    agent=agent_hi,
    max_t=320,
    seed=VIZ_SEED,
    wrappers=[RGBImgObsWrapper, lambda env: HookStepEnv(env, agent_hi)],
)

## Demo: language-guided rollout

We pick a mission string, find a seed that generates it, and run the policy with a high desired
return-to-go. This gives a quick qualitative check that the model follows language as well as return
conditioning.


MiniGrid mission strings can vary across library versions — e.g., `"go to the green ball"` vs.
`"go to a green ball"`. To make our seed search robust, `_mission_signature` extracts just the
`(color, object_type)` pair and matches on that instead of the exact string.

In [ ]:
def _mission_signature(m: str) -> Tuple[str, str]:
    """Return (color, obj_type) from common MiniGrid GoTo* mission strings.

    Works across versions that differ in articles, e.g.:
      - 'go to the green ball'
      - 'go to green ball'
      - 'go to a green ball'
    """
    m = " ".join(m.lower().strip().split())
    toks = m.split()
    if len(toks) < 2:
        return ("", "")
    return (toks[-2], toks[-1])


def find_seed_for_mission(
    target_mission: str, max_tries: int = 5000, start_seed: int = 0
) -> Optional[int]:
    """Find a seed that yields `target_mission` under `gym_simulation`'s reset pattern."""
    tmp = make_env()
    target_sig = _mission_signature(target_mission)
    try:
        for s in range(start_seed, start_seed + max_tries):
            obs, _ = tmp.reset(seed=s)
            if _mission_signature(obs["mission"]) == target_sig:
                return s
    finally:
        tmp.close()
    return None


# ---- pick a mission sentence and run the demo ----
DEMO_MISSION = "go to the yellow ball"  # try changing this!
seed = find_seed_for_mission(DEMO_MISSION)

if seed is None:
    print(f"Couldn't find '{DEMO_MISSION}' quickly; falling back to seed=0.")
    seed = 0

# Sanity-check: what mission does the env actually generate at this seed?
_tmp = make_env()
_tmp.reset(seed=seed)  # init_random(...)
obs, _ = _tmp.reset(seed=seed)  # gym_simulation reset(...)
_tmp.close()

print(f"Demo mission request: '{DEMO_MISSION}'")
print(f"Env mission at seed={seed}: '{obs['mission']}'")

print(f"Using seed={seed}, desired_return={hi:.3f}")
demo_agent = DTVLAAgentAdapter(model, desired_return=hi)
gym_simulation(
    ENV_ID,
    agent=demo_agent,
    max_t=320,
    seed=seed,
    wrappers=[RGBImgObsWrapper, lambda env: HookStepEnv(env, demo_agent)],
)

## Frontiers

### What our NanoVLA _doesn't_ do (vs. real systems)

Our nano setup is deliberately minimal. Here's what production VLAs and Decision Transformers add:

- **Continuous actions.** Real robot policies output joint torques or end-effector deltas
  (regression heads, often with diffusion or GMM decoders), not discrete grid actions.
- **Pretrained backbones.** Systems like RT-2 and OpenVLA use frozen CLIP/ViT image encoders and
  large language models, not a 3-layer CNN and a whitespace tokenizer.
- **Scale.** Our model has a few million parameters; OpenVLA has 7B+. Data goes from our 2,500
  synthetic episodes to millions of real-robot demonstrations.
- **Positional encoding.** We use simple learned per-timestep embeddings; real transformers often
  use sinusoidal, RoPE, or ALiBi schemes that generalize to unseen sequence lengths.

### DT limitations and extensions

The original Decision Transformer shows that sequence modeling is a viable alternative to
value-based offline RL, but it has known gaps:

- **No "stitching":** DT can only reproduce behaviors it has seen in contiguous trajectory windows.
  Unlike IQL or CQL, it cannot combine the best parts of different trajectories to synthesize a
  better-than-demonstrated policy. **Elastic DT** addresses this by allowing variable-length context
  and more flexible attention patterns.
- **Hindsight return relabeling:** if we relabel RTG values after collection (e.g., by computing
  RTG from a learned value function rather than raw rewards), we can improve conditioning quality
  without collecting new data.
- **Online fine-tuning:** **Online DT** extends the framework by alternating between collecting new
  data with the current policy and retraining the DT on the growing dataset — bridging the gap
  between offline and online RL.

### From discrete to continuous actions

Our NanoVLA predicts discrete actions via a simple classification head (softmax + argmax). Real
robotic VLAs must output **continuous** control signals (joint angles, velocities, end-effector
poses). This is harder because the action space is high-dimensional and multimodal — there may be
multiple valid ways to reach the same goal. Two popular approaches:

- **Diffusion heads** (used by Octo, Diffusion Policy): model the action distribution as a
  denoising diffusion process. At inference time, the head iteratively refines random noise into a
  coherent action sequence. This handles multimodality naturally.
- **GMM decoders** (Gaussian Mixture Models): predict a mixture of Gaussians over the action space,
  then sample from the most likely mode. Simpler than diffusion but less expressive for complex
  distributions.

### Papers to explore next

- RT-2 (VLA transfer from web knowledge)
- OpenVLA (open-source generalist VLA)
- VIMA (multimodal prompts benchmark + transformer agent)
- Octo (generalist robot policies; diffusion heads)
- Physical Intelligence $\pi_0$ (VLA learning from experience with RL signals)

These are _far_ beyond the nano setup here, but the core theme is similar: **treat control as
sequence modeling over multimodal tokens**, plus better data, better pretraining, and better
fine-tuning.